# Графики для отчёта

Короткий notebook для построения финальных рисунков из уже обработанных данных проекта. Он не запускает MD и не пересчитывает исходные CSV.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()
PROCESSED_DIR = PROJECT_DIR / 'local_data' / 'processed'
FIGURES_DIR = PROJECT_DIR / 'графики'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'legend.frameon': False,
})

print('PROCESSED_DIR:', PROCESSED_DIR)
print('FIGURES_DIR:', FIGURES_DIR)


In [ ]:
eos = pd.read_csv(PROCESSED_DIR / 'eos_mean_by_T_rho.csv')
fit = json.loads((PROCESSED_DIR / 'vdw_fit_warm_gas.json').read_text(encoding='utf-8'))
spinodal_points = pd.read_csv(PROCESSED_DIR / 'approx_spinodal_points_for_plot.csv')
vdw_spinodal = pd.read_csv(PROCESSED_DIR / 'vdw_spinodal_curve.csv')
vdw_binodal = pd.read_csv(PROCESSED_DIR / 'vdw_binodal_curve.csv')

a_fit = float(fit['a'])
b_fit = float(fit['b'])

def vdw_pressure(rho, T):
    return rho * T / (1.0 - b_fit * rho) - a_fit * rho * rho

def take_temps(df, temps):
    parts = []
    available = np.sort(df['T_target'].dropna().unique())
    for T in temps:
        idx = int(np.argmin(np.abs(available - T)))
        T_found = float(available[idx])
        if not np.isclose(T_found, T, atol=1e-8):
            print(f'Предупреждение: T={T:g} не найдена, использую T={T_found:g}')
        parts.append((T_found, df[np.isclose(df['T_target'], T_found)].copy()))
    return parts

def save_current(name):
    path = FIGURES_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches='tight')
    plt.show()
    print('saved:', path)

print('VdW a =', a_fit)
print('VdW b =', b_fit)
print('T:', sorted(eos['T_target'].unique()))


## Давление

In [ ]:
pressure_temps = [1.50, 1.30, 1.10, 0.95, 0.80]

plt.figure(figsize=(7.0, 4.6))
for T, part in take_temps(eos, pressure_temps):
    part = part.sort_values('rho_target')
    plt.errorbar(
        part['rho_target'], part['P_mean'], yerr=part['P_err'],
        marker='o', markersize=3.2, linewidth=1.4, capsize=2,
        label=f'T={T:.2f}',
    )

plt.xlabel(r'$\rho$')
plt.ylabel(r'$P$')
plt.title('Давление')
plt.legend(ncol=2, fontsize=9)
save_current('давление_от_плотности.png')


## Давление при малых плотностях


In [ ]:
pressure_low_rho_temps = [1.50, 1.30, 1.10, 0.95, 0.80]

plt.figure(figsize=(7.0, 4.6))
for T, part in take_temps(eos, pressure_low_rho_temps):
    part = part.sort_values('rho_target')
    low = part[part['rho_target'] <= 0.3].copy()
    plt.errorbar(
        low['rho_target'], low['P_mean'], yerr=low['P_err'],
        marker='o', markersize=3.2, linewidth=1.4, capsize=2,
        label=f'T={T:.2f}',
    )

plt.axhline(0.0, color='black', linewidth=1.0)
plt.xlabel(r'$\rho$')
plt.ylabel(r'$P$')
plt.title(r'Давление, $\rho\leq0.3$')
plt.xlim(left=0.0, right=0.305)
plt.legend(ncol=2, fontsize=9)
save_current('давление_малые_плотности.png')


## Остатки VdW

In [ ]:
residual_temps = [1.50, 1.30, 1.10, 0.90]

plt.figure(figsize=(7.0, 4.4))
plt.axhline(0.0, color='black', linewidth=1.0)
for T, part in take_temps(eos, residual_temps):
    part = part.sort_values('rho_target').copy()
    part['P_residual'] = part['P_mean'] - vdw_pressure(part['rho_target'].to_numpy(dtype=float), T)
    plt.plot(
        part['rho_target'], part['P_residual'],
        marker='o', markersize=3.2, linewidth=1.4,
        label=f'T={T:.2f}',
    )

plt.xlabel(r'$\rho$')
plt.ylabel(r'$P-P_{\mathrm{VdW}}$')
plt.title('Остатки VdW')
plt.legend(ncol=2, fontsize=9)
save_current('остатки_vdw.png')


## Остатки VdW при малых плотностях


In [ ]:
residual_low_rho_temps = [1.50, 1.30, 1.10, 0.90]

plt.figure(figsize=(7.0, 4.4))
plt.axhline(0.0, color='black', linewidth=1.0)
for T, part in take_temps(eos, residual_low_rho_temps):
    part = part.sort_values('rho_target').copy()
    low = part[part['rho_target'] <= 0.4].copy()
    low['P_residual'] = low['P_mean'] - vdw_pressure(low['rho_target'].to_numpy(dtype=float), T)
    plt.plot(
        low['rho_target'], low['P_residual'],
        marker='o', markersize=3.2, linewidth=1.4,
        label=f'T={T:.2f}',
    )

plt.xlabel(r'$\rho$')
plt.ylabel(r'$P-P_{\mathrm{VdW}}$')
plt.title(r'Остатки VdW, $\rho\leq0.4$')
plt.xlim(left=0.0, right=0.405)
plt.legend(ncol=2, fontsize=9)
save_current('остатки_vdw_малые_плотности.png')


## Спинодаль и бинодаль

In [ ]:
plt.figure(figsize=(6.4, 4.8))
plt.plot(vdw_spinodal['rho'], vdw_spinodal['T_vdw_spinodal'], color='black', linewidth=1.8, label='VdW спинодаль')

if len(vdw_binodal):
    plt.plot(vdw_binodal['rho_gas'], vdw_binodal['T'], color='black', linestyle='--', linewidth=1.5, label='VdW бинодаль')
    plt.plot(vdw_binodal['rho_liquid'], vdw_binodal['T'], color='black', linestyle='--', linewidth=1.5)

gas = spinodal_points[spinodal_points['branch'] == 'gas'].sort_values('rho_target')
dense = spinodal_points[spinodal_points['branch'] == 'dense'].sort_values('rho_target')
plt.scatter(gas['rho_target'], gas['T_target'], s=42, marker='o', label='MD газ')
plt.scatter(dense['rho_target'], dense['T_target'], s=42, marker='s', label='MD жидкость')

plt.xlabel(r'$\rho$')
plt.ylabel(r'$T$')
plt.title('Спинодаль и бинодаль')
plt.xlim(left=0.0)
plt.ylim(bottom=0.0)
plt.legend(fontsize=9)
save_current('спинодаль_бинодаль_vdw.png')


## Потенциальная энергия

In [ ]:
energy_temps = [0.70, 0.80, 0.90, 1.00, 1.30, 1.50]

plt.figure(figsize=(7.0, 4.6))
for T, part in take_temps(eos, energy_temps):
    part = part.sort_values('rho_target')
    plt.plot(
        part['rho_target'], part['U_mean'],
        marker='o', markersize=3.2, linewidth=1.4,
        label=f'T={T:.2f}',
    )

plt.xlabel(r'$\rho$')
plt.ylabel(r'$U/N$')
plt.title('Энергия')
plt.legend(ncol=2, fontsize=9)
save_current('энергия_на_частицу.png')


## Файлы

In [ ]:
print('Готовые графики:')
for path in sorted(FIGURES_DIR.glob('*.png')):
    print(path.name)
